In [ ]:
"""
Zonal vertical cross-sections of potential vorticity along the latitude of P1
(42.5 N), together with 2-PVU potential-temperature maps.

Left column : theta on the dynamical tropopause (2 PVU) with T45 theta contours
              and MSLP (contours + lows). The red line marks the cross-section
              (42.5 N).
Right column: west-east vertical PV cross-section along 42.5 N. The dashed line
              marks the longitude of P1 (10 W).

The figure is an N x 2 panel grid (N = number of dates) with (a), (b), ...
panel labels and a shared colorbar per column.
"""

import os
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.ticker import FixedLocator, FuncFormatter, ScalarFormatter
from scipy.ndimage import gaussian_filter, minimum_filter
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================================
# Paths & configuration
# =========================================================================
base_dir = "./06_derived_data/rwb_era5/"
fn_theta = os.path.join(base_dir, "theta_2pvu_20260510_20260601.nc")
fn_theta_T45 = os.path.join(base_dir, "theta_2pvu_20260510_20260601_T45.nc")
fn_mslp_nc = os.path.join(base_dir, "mslp_20260510_20260601.nc")

pv_dir = "/pool/data/ERA5/ET/pl/an/1H/060/"

# Map domain
lon_min, lon_max = -60, 20
lat_min, lat_max = 20, 75

# Contour / fill levels
theta_levels_fill = np.arange(270, 370, 2)
theta_levels_trunc = np.arange(280, 370, 4)
pv_levels = np.arange(0, 10.5, 0.5)

# MSLP settings
mslp_low_max_hpa = 1000            # only plot contours / mark lows at or below this value
mslp_contour_interval_hpa = 4      # spacing of MSLP contours
mslp_levels = np.arange(940, mslp_low_max_hpa + mslp_contour_interval_hpa,
                        mslp_contour_interval_hpa)

# Reference points
pt1_lon, pt1_lat = -10.0, 42.5        # P1
pt_paris_lon, pt_paris_lat = 2.35, 48.85  # Paris (map reference only)

# Dates to plot (one row each)
target_dates = [
    "2026-05-19T06:00:00",
    "2026-05-21T18:00:00",
    "2026-05-23T00:00:00",
    "2026-05-24T00:00:00",
]

# =========================================================================
# Cross-section geometry: ZONAL section along the latitude of P1 (42.5 N)
# =========================================================================
cross_lat = pt1_lat                    # 42.5 N
cross_lon_min = pt1_lon - 30.0         # -40
cross_lon_max = pt1_lon + 20.0         # +10
num_pts = 300

transect_lons = np.linspace(cross_lon_min, cross_lon_max, num_pts)
transect_lats = np.full(num_pts, cross_lat)

# =========================================================================
# Global font settings for a publishable figure
# =========================================================================
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 15,
    "axes.labelsize": 14,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "axes.linewidth": 0.8,
})


def lon_formatter(x, _pos):
    """Format a longitude value as e.g. 10 W / 5 E / 0."""
    x = int(round(x))
    if x < 0:
        return f"{abs(x)}°W"
    if x > 0:
        return f"{x}°E"
    return "0°"


def add_panel_label(ax, label):
    """Bold panel letter, outside the axes at the top-left."""
    ax.text(
        -0.07, 1.03, label,
        transform=ax.transAxes,
        fontsize=17, fontweight="bold",
        ha="left", va="bottom", clip_on=False,
    )


# =========================================================================
# Grid & coordinate helpers
# =========================================================================
def fix_longitude_coord(data_obj):
    lon_name = "longitude" if "longitude" in data_obj.coords else "lon"
    lat_name = "latitude" if "latitude" in data_obj.coords else "lat"
    if data_obj[lon_name].max() > 180:
        new_lons = ((data_obj[lon_name] + 180) % 360) - 180
        data_obj = data_obj.assign_coords({lon_name: new_lons})
    if lat_name in data_obj.coords and lon_name in data_obj.coords:
        data_obj = data_obj.sortby(lat_name).sortby(lon_name)
    return data_obj, lon_name, lat_name


def slice_spatial_domain(obj, lon0, lon1, lat0, lat1, lon_name, lat_name):
    obj = obj.sortby(lat_name, ascending=True).sortby(lon_name, ascending=True)
    mask = (
        (obj[lon_name] >= lon0) & (obj[lon_name] <= lon1) &
        (obj[lat_name] >= lat0) & (obj[lat_name] <= lat1)
    )
    return obj.where(mask, drop=True)


# =========================================================================
# Load NetCDF data
# =========================================================================
ds_theta = xr.open_dataset(fn_theta)
da_theta = ds_theta["pt"] if "pt" in ds_theta.data_vars else ds_theta["theta"]
da_theta, theta_lon_name, theta_lat_name = fix_longitude_coord(da_theta)

ds_theta_T45 = xr.open_dataset(fn_theta_T45)
da_theta_T45 = ds_theta_T45["pt"] if "pt" in ds_theta_T45.data_vars else ds_theta_T45["theta"]
da_theta_T45, theta_T45_lon_name, theta_T45_lat_name = fix_longitude_coord(da_theta_T45)

ds_mslp_nc = xr.open_dataset(fn_mslp_nc)
da_mslp_nc = ds_mslp_nc["msl"] if "msl" in ds_mslp_nc.data_vars else ds_mslp_nc["mslp"]
da_mslp_nc, msl_nc_lon_name, msl_nc_lat_name = fix_longitude_coord(da_mslp_nc)


# =========================================================================
# Per-date panel plotting
# =========================================================================
def plot_map(ax_map, date_str, bottom_labels=True):
    """Left panel: theta on 2 PVU + T45 contours + MSLP lows + section line."""
    dt = pd.to_datetime(date_str)

    theta_sub = slice_spatial_domain(
        da_theta.sel(time=date_str).squeeze(),
        lon_min, lon_max, lat_min, lat_max, theta_lon_name, theta_lat_name,
    )
    cf = ax_map.contourf(
        theta_sub[theta_lon_name], theta_sub[theta_lat_name], theta_sub.values,
        levels=theta_levels_fill, cmap="viridis", extend="both",
        transform=ccrs.PlateCarree(),
    )

    # T45 theta contours + MSLP (contours + lows) from the NetCDF files
    try:
        theta_T45_sub = slice_spatial_domain(
            da_theta_T45.sel(time=date_str).squeeze(),
            lon_min, lon_max, lat_min, lat_max,
            theta_T45_lon_name, theta_T45_lat_name,
        )
        cs_trunc = ax_map.contour(
            theta_T45_sub[theta_T45_lon_name], theta_T45_sub[theta_T45_lat_name],
            gaussian_filter(theta_T45_sub.values, sigma=1.0),
            levels=theta_levels_trunc, colors="black", linewidths=0.9,
            transform=ccrs.PlateCarree(),
        )
        ax_map.clabel(cs_trunc, fmt="%d", inline=True, fontsize=9)

        msl_nc_sub = slice_spatial_domain(
            da_mslp_nc.sel(time=date_str).squeeze(),
            lon_min, lon_max, lat_min, lat_max,
            msl_nc_lon_name, msl_nc_lat_name,
        ) / 100.0
        lat_name_msl = "latitude" if "latitude" in msl_nc_sub.coords else "lat"
        msl_2d = msl_nc_sub.values

        # MSLP contours (from the NetCDF field, same source as the L markers)
        cs_msl = ax_map.contour(
            msl_nc_sub[msl_nc_lon_name], msl_nc_sub[lat_name_msl], msl_2d,
            levels=mslp_levels, colors="white", linewidths=1.5, linestyles="dashed",
            transform=ccrs.PlateCarree(),
        )
        ax_map.clabel(cs_msl, fmt="%d", inline=True, fontsize=9)

        # MSLP lows
        #local_min = (minimum_filter(msl_2d, size=12) == msl_2d) & (msl_2d < mslp_low_max_hpa)
        #for r_idx, c_idx in zip(*np.where(local_min)):
        #    ax_map.text(
        #        msl_nc_sub[msl_nc_lon_name][c_idx], msl_nc_sub[lat_name_msl][r_idx],
        #        f"L\n{int(msl_2d[r_idx, c_idx])}",
        #        color="white", weight="bold", fontsize=9, ha="center", va="center",
        #        transform=ccrs.PlateCarree(),
        #    )
    except Exception as exc:  # noqa: BLE001
        print(f"Warning adding T45/MSLP features for {date_str}: {exc}")

    # Cross-section line (along 42.5 N), P1 and Paris markers
    ax_map.plot(
        [cross_lon_min, cross_lon_max], [cross_lat, cross_lat],
        color="red", linewidth=2.0, transform=ccrs.PlateCarree(),
    )
    ax_map.plot(pt1_lon, pt1_lat, "o", color="red", markersize=8,
                markeredgecolor="white", transform=ccrs.PlateCarree())
    ax_map.text(
        pt1_lon + 1.0, pt1_lat - 2.2, "P1",
        color="white", weight="bold", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6),
        transform=ccrs.PlateCarree(),
    )
    ax_map.plot(pt_paris_lon, pt_paris_lat, "o", color="yellow", markersize=7,
                markeredgecolor="black", transform=ccrs.PlateCarree())
    ax_map.text(
        pt_paris_lon + 1.0, pt_paris_lat + 1.0, "Paris",
        color="yellow", weight="bold", fontsize=12,
        bbox=dict(boxstyle="round,pad=0.2", facecolor="black", alpha=0.6),
        transform=ccrs.PlateCarree(),
    )

    ax_map.coastlines(linewidth=0.7)
    ax_map.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax_map.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # Coordinate gridlines / labels
    gl = ax_map.gridlines(
        draw_labels=True, linewidth=0.4, color="gray", alpha=0.5, linestyle="--",
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = True
    gl.bottom_labels = bottom_labels
    gl.xlocator = FixedLocator(np.arange(-60, 21, 20))
    gl.ylocator = FixedLocator(np.arange(30, 76, 15))
    gl.xlabel_style = {"size": 11}
    gl.ylabel_style = {"size": 11}

    ax_map.set_title(f"θ at 2 PVU  |  {dt.strftime('%Y-%m-%d %H:%M UTC')}")
    return cf


def plot_cross(ax_cross, date_str):
    """Right panel: zonal vertical PV cross-section along 42.5 N."""
    dt = pd.to_datetime(date_str)
    pv_filepath = os.path.join(pv_dir, f"ETpl00_1H_{dt.strftime('%Y-%m-%d')}_060.grb")

    if not os.path.exists(pv_filepath):
        ax_cross.text(0.5, 0.5, "PV file not found", ha="center", va="center",
                      transform=ax_cross.transAxes)
        return None

    try:
        ds_pv = xr.open_dataset(
            pv_filepath, engine="cfgrib",
            backend_kwargs={
                "indexpath": "",
                "filter_by_keys": {"shortName": "pv", "typeOfLevel": "isobaricInhPa"},
            },
        )
        ds_pv, pv_lon_name, pv_lat_name = fix_longitude_coord(ds_pv)

        plevs = sorted(p for p in ds_pv.coords["isobaricInhPa"].values if 150 <= p <= 850)
        pv_da = ds_pv["pv"].sel(time=date_str, isobaricInhPa=plevs)

        # Nearest-neighbour sampling of the (unstructured) grid onto the transect
        tree = cKDTree(np.column_stack((ds_pv[pv_lon_name].values,
                                        ds_pv[pv_lat_name].values)))
        _, indices = tree.query(np.column_stack((transect_lons, transect_lats)))
        pv_cross = pv_da.values[:, indices] * 1e6  # -> PVU

        cf_pv = ax_cross.contourf(
            transect_lons, plevs, pv_cross,
            levels=pv_levels, cmap="YlOrRd", extend="max",
        )
        cs_2pvu = ax_cross.contour(
            transect_lons, plevs, pv_cross, levels=[2.0],
            colors="black", linewidths=2.0,
        )
        ax_cross.clabel(cs_2pvu, fmt="2 PVU", inline=True, fontsize=9)

        # Mark the longitude of P1
        ax_cross.axvline(pt1_lon, color="blue", linestyle="--", linewidth=1.5, alpha=0.8)
        #ax_cross.text(pt1_lon, 165, " P1", color="blue", weight="bold", fontsize=11)

        ax_cross.set_yscale("log")
        ax_cross.set_yticks([850, 700, 500, 400, 300, 200])
        ax_cross.get_yaxis().set_major_formatter(ScalarFormatter())
        ax_cross.invert_yaxis()
        ax_cross.set_ylim(850, 150)

        ax_cross.set_xlim(cross_lon_min, cross_lon_max)
        ax_cross.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
        ax_cross.set_xlabel("")
        ax_cross.set_ylabel("Pressure [hPa]")
        ax_cross.set_title(f"42.5°N,  {dt.strftime('%Y-%m-%d %H:%M UTC')}")
        return cf_pv

    except Exception as exc:  # noqa: BLE001
        ax_cross.text(0.5, 0.5, f"Error:\n{exc}", ha="center", va="center",
                      transform=ax_cross.transAxes, wrap=True)
        print(f"Error for {date_str}: {exc}")
        return None


# =========================================================================
# Build the N x 2 figure
# =========================================================================
n_rows = len(target_dates)
fig = plt.figure(figsize=(14, 4 * n_rows))
gs = fig.add_gridspec(
    n_rows, 2, width_ratios=[1.0, 1.0],
    left=0.06, right=0.97, bottom=0.08, top=0.95,
    hspace=0.30, wspace=0.14,
)

panel_labels = [f"({c})" for c in "abcdefghijklmnop"]
cf_map = cf_pv = None

for row, date_str in enumerate(target_dates):
    ax_map = fig.add_subplot(gs[row, 0], projection=ccrs.PlateCarree())
    ax_cross = fig.add_subplot(gs[row, 1])

    m = plot_map(ax_map, date_str, bottom_labels=(row == n_rows - 1))
    c = plot_cross(ax_cross, date_str)
    cf_map = m if m is not None else cf_map
    cf_pv = c if c is not None else cf_pv

    add_panel_label(ax_map, panel_labels[2 * row])
    add_panel_label(ax_cross, panel_labels[2 * row + 1])

# Shared colorbars, one per column
if cf_map is not None:
    cax1 = fig.add_axes([0.10, 0.040, 0.30, 0.010])
    cb1 = fig.colorbar(cf_map, cax=cax1, orientation="horizontal")
    cb1.set_label("Potential temperature [K]")

if cf_pv is not None:
    cax2 = fig.add_axes([0.60, 0.040, 0.30, 0.010])
    cb2 = fig.colorbar(cf_pv, cax=cax2, orientation="horizontal")
    cb2.set_label("Potential vorticity [PVU]")

fig.savefig("pv_zonal_section_p1.png", dpi=200, bbox_inches="tight")
fig.savefig("pv_zonal_section_p1.pdf", bbox_inches="tight")
plt.show()


In [ ]:
"""
Zonal vertical cross-sections of potential vorticity along the latitude of P1
(42.5 N), together with 2-PVU potential-temperature maps.

Top row    : theta on the dynamical tropopause (2 PVU) with T45 theta contours
             and MSLP contours. The red line marks the cross-section (42.5 N);
             red dot = P1, yellow dot = Paris.
Bottom row : west-east vertical PV cross-section along 42.5 N. The dashed line
             marks the longitude of P1 (10 W).

Colormaps: discrete colours sampled from the PV colorbar of Ezber (2026).
Bottom row uses them with the Ezber PV levels (0.5-12 PVU, blue below and
pink/purple above 2 PVU). Top row uses the same colours reversed, switching
from pink/purple (low theta, northern air) to blue (high theta, southern
air) at theta_cut = 340 K.

The figure is a 2 x N panel grid (N = number of dates, time running left to
right). Panels (a)-(d) are the maps, (e)-(h) the cross-sections; one colorbar
per row.
"""

import os
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, ScalarFormatter
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)

# =========================================================================
# Paths & configuration
# =========================================================================
base_dir = "./06_derived_data/rwb_era5/"
fn_theta = os.path.join(base_dir, "theta_2pvu_20260510_20260601.nc")
fn_theta_T45 = os.path.join(base_dir, "theta_2pvu_20260510_20260601_T45.nc")
fn_mslp_nc = os.path.join(base_dir, "mslp_20260510_20260601.nc")

pv_dir = "/pool/data/ERA5/ET/pl/an/1H/060/"

# Map domain
lon_min, lon_max = -60, 20
lat_min, lat_max = 20, 75

# Contour / fill levels
theta_levels_trunc = np.arange(280, 370, 4)

# Colours sampled from the PV colorbar of Ezber (2026), low -> high PV:
# blues (0.5-2 PVU, = blue half of Crameri "vik") and pinks/purples (2-12 PVU).
ezber_colors = [
    "#cce1ec", "#94bbd2", "#458eb5",                          # 0.5-1, 1-1.5, 1.5-2
    "#e7789b", "#d94a8d", "#b8237a", "#8e256d", "#652760", "#3b2155",  # 2-3 ... 10-12
]

# Row 1: same colours reversed, blue/pink switch at theta_cut.
# < theta_cut: pinks darkening to purple towards low theta (northern air);
# > theta_cut: blues fading to white towards high theta (southern air).
theta_cut, theta_step = 335, 5
theta_levels_fill = theta_cut + theta_step * np.arange(-6, 4)   # 310 ... 355 K
theta_colors = [ezber_colors[-1]] + ezber_colors[::-1] + ["white"]  # under, 9 bins, over

# Row 2: Ezber PV levels; < 0.5 PVU unshaded, > 12 PVU darkest purple
pv_levels = [0.5, 1, 1.5, 2, 3, 4, 6, 8, 10, 12]
pv_colors = ezber_colors + [ezber_colors[-1]]                    # 9 bins, over

# MSLP settings
mslp_low_max_hpa = 1000            # only plot contours at or below this value
mslp_contour_interval_hpa = 4      # spacing of MSLP contours
mslp_levels = np.arange(940, mslp_low_max_hpa + mslp_contour_interval_hpa,
                        mslp_contour_interval_hpa)

# Reference points
pt1_lon, pt1_lat = -10.0, 42.5        # P1
pt_paris_lon, pt_paris_lat = 2.35, 48.85  # Paris (map reference only)

# Dates to plot (one column each)
target_dates = [
    "2026-05-19T06:00:00",
    "2026-05-21T18:00:00",
    "2026-05-23T00:00:00",
    "2026-05-24T00:00:00",
]

# =========================================================================
# Cross-section geometry: ZONAL section along the latitude of P1 (42.5 N)
# =========================================================================
cross_lat = pt1_lat                    # 42.5 N
cross_lon_min = pt1_lon - 30.0         # -40
cross_lon_max = pt1_lon + 20.0         # +10
num_pts = 300

transect_lons = np.linspace(cross_lon_min, cross_lon_max, num_pts)
transect_lats = np.full(num_pts, cross_lat)

# =========================================================================
# Global font settings (figure is 12 in wide, scaled to ~17 cm in print)
# =========================================================================
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10.5,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.7,
})


def lon_formatter(x, _pos):
    """Format a longitude value as e.g. 10 W / 5 E / 0."""
    x = int(round(x))
    if x < 0:
        return f"{abs(x)}°W"
    if x > 0:
        return f"{x}°E"
    return "0°"


def add_panel_label(ax, label):
    """Bold panel letter, outside the axes at the top-left."""
    ax.text(
        -0.07, 1.03, label,
        transform=ax.transAxes,
        fontsize=12, fontweight="bold",
        ha="left", va="bottom", clip_on=False,
    )


# =========================================================================
# Grid & coordinate helpers
# =========================================================================
def fix_longitude_coord(data_obj):
    lon_name = "longitude" if "longitude" in data_obj.coords else "lon"
    lat_name = "latitude" if "latitude" in data_obj.coords else "lat"
    if data_obj[lon_name].max() > 180:
        new_lons = ((data_obj[lon_name] + 180) % 360) - 180
        data_obj = data_obj.assign_coords({lon_name: new_lons})
    if lat_name in data_obj.coords and lon_name in data_obj.coords:
        data_obj = data_obj.sortby(lat_name).sortby(lon_name)
    return data_obj, lon_name, lat_name


def slice_spatial_domain(obj, lon0, lon1, lat0, lat1, lon_name, lat_name):
    obj = obj.sortby(lat_name, ascending=True).sortby(lon_name, ascending=True)
    mask = (
        (obj[lon_name] >= lon0) & (obj[lon_name] <= lon1) &
        (obj[lat_name] >= lat0) & (obj[lat_name] <= lat1)
    )
    return obj.where(mask, drop=True)


# =========================================================================
# Load NetCDF data
# =========================================================================
ds_theta = xr.open_dataset(fn_theta)
da_theta = ds_theta["pt"] if "pt" in ds_theta.data_vars else ds_theta["theta"]
da_theta, theta_lon_name, theta_lat_name = fix_longitude_coord(da_theta)

ds_theta_T45 = xr.open_dataset(fn_theta_T45)
da_theta_T45 = ds_theta_T45["pt"] if "pt" in ds_theta_T45.data_vars else ds_theta_T45["theta"]
da_theta_T45, theta_T45_lon_name, theta_T45_lat_name = fix_longitude_coord(da_theta_T45)

ds_mslp_nc = xr.open_dataset(fn_mslp_nc)
da_mslp_nc = ds_mslp_nc["msl"] if "msl" in ds_mslp_nc.data_vars else ds_mslp_nc["mslp"]
da_mslp_nc, msl_nc_lon_name, msl_nc_lat_name = fix_longitude_coord(da_mslp_nc)


# =========================================================================
# Per-date panel plotting
# =========================================================================
def plot_map(ax_map, date_str, left_labels=True):
    """Top panel: theta on 2 PVU + T45 contours + MSLP + section line."""
    dt = pd.to_datetime(date_str)

    theta_sub = slice_spatial_domain(
        da_theta.sel(time=date_str).squeeze(),
        lon_min, lon_max, lat_min, lat_max, theta_lon_name, theta_lat_name,
    )
    cf = ax_map.contourf(
        theta_sub[theta_lon_name], theta_sub[theta_lat_name], theta_sub.values,
        levels=theta_levels_fill, colors=theta_colors, extend="both",
        transform=ccrs.PlateCarree(),
    )

    # T45 theta contours + MSLP contours from the NetCDF files
    try:
        theta_T45_sub = slice_spatial_domain(
            da_theta_T45.sel(time=date_str).squeeze(),
            lon_min, lon_max, lat_min, lat_max,
            theta_T45_lon_name, theta_T45_lat_name,
        )
        cs_trunc = ax_map.contour(
            theta_T45_sub[theta_T45_lon_name], theta_T45_sub[theta_T45_lat_name],
            gaussian_filter(theta_T45_sub.values, sigma=1.0),
            levels=theta_levels_trunc, colors="black", linewidths=0.6,
            transform=ccrs.PlateCarree(),
        )
        ax_map.clabel(cs_trunc, fmt="%d", inline=True, fontsize=6)

        msl_nc_sub = slice_spatial_domain(
            da_mslp_nc.sel(time=date_str).squeeze(),
            lon_min, lon_max, lat_min, lat_max,
            msl_nc_lon_name, msl_nc_lat_name,
        ) / 100.0
        lat_name_msl = "latitude" if "latitude" in msl_nc_sub.coords else "lat"

        cs_msl = ax_map.contour(
            msl_nc_sub[msl_nc_lon_name], msl_nc_sub[lat_name_msl], msl_nc_sub.values,
            levels=mslp_levels, colors="white", linewidths=1.0, linestyles="dashed",
            transform=ccrs.PlateCarree(),
        )
        ax_map.clabel(cs_msl, fmt="%d", inline=True, fontsize=6)
    except Exception as exc:  # noqa: BLE001
        print(f"Warning adding T45/MSLP features for {date_str}: {exc}")

    # Cross-section line (along 42.5 N), P1 (red dot) and Paris (yellow dot)
    ax_map.plot(
        [cross_lon_min, cross_lon_max], [cross_lat, cross_lat],
        color="red", linewidth=1.5, transform=ccrs.PlateCarree(),
        path_effects=[pe.Stroke(linewidth=2.6, foreground="white"), pe.Normal()],
    )
    ax_map.plot(pt1_lon, pt1_lat, "o", color="red", markersize=5,
                markeredgecolor="white", markeredgewidth=0.8,
                transform=ccrs.PlateCarree(), zorder=5)
    ax_map.plot(pt_paris_lon, pt_paris_lat, "o", color="yellow", markersize=5,
                markeredgecolor="black", markeredgewidth=0.8,
                transform=ccrs.PlateCarree(), zorder=5)

    ax_map.coastlines(linewidth=0.6)
    ax_map.add_feature(cfeature.BORDERS, linewidth=0.3)
    ax_map.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())

    # Coordinate gridlines / labels
    gl = ax_map.gridlines(
        draw_labels=True, linewidth=0.4, color="gray", alpha=0.5, linestyle="--",
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = left_labels
    gl.bottom_labels = True
    gl.xlocator = FixedLocator(np.arange(-60, 21, 20))
    gl.ylocator = FixedLocator(np.arange(30, 76, 15))
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}

    ax_map.set_title(dt.strftime("%d %b %Y %H UTC"))
    return cf


def plot_cross(ax_cross, date_str, left_labels=True):
    """Bottom panel: zonal vertical PV cross-section along 42.5 N."""
    dt = pd.to_datetime(date_str)
    pv_filepath = os.path.join(pv_dir, f"ETpl00_1H_{dt.strftime('%Y-%m-%d')}_060.grb")

    if not os.path.exists(pv_filepath):
        ax_cross.text(0.5, 0.5, "PV file not found", ha="center", va="center",
                      transform=ax_cross.transAxes)
        return None

    try:
        ds_pv = xr.open_dataset(
            pv_filepath, engine="cfgrib",
            backend_kwargs={
                "indexpath": "",
                "filter_by_keys": {"shortName": "pv", "typeOfLevel": "isobaricInhPa"},
            },
        )
        ds_pv, pv_lon_name, pv_lat_name = fix_longitude_coord(ds_pv)

        plevs = sorted(p for p in ds_pv.coords["isobaricInhPa"].values if 150 <= p <= 850)
        pv_da = ds_pv["pv"].sel(time=date_str, isobaricInhPa=plevs)

        # Nearest-neighbour sampling of the (unstructured) grid onto the transect
        tree = cKDTree(np.column_stack((ds_pv[pv_lon_name].values,
                                        ds_pv[pv_lat_name].values)))
        _, indices = tree.query(np.column_stack((transect_lons, transect_lats)))
        pv_cross = pv_da.values[:, indices] * 1e6  # -> PVU

        cf_pv = ax_cross.contourf(
            transect_lons, plevs, pv_cross,
            levels=pv_levels, colors=pv_colors, extend="max",
        )
        cs_2pvu = ax_cross.contour(
            transect_lons, plevs, pv_cross, levels=[2.0],
            colors="black", linewidths=1.5,
        )
        ax_cross.clabel(cs_2pvu, fmt="2 PVU", inline=True, fontsize=7)

        # Mark the longitude of P1
        ax_cross.axvline(pt1_lon, color="black", linestyle="--", linewidth=1.2)

        # Log-pressure axis; minor-tick labels off (avoids "6x10^2")
        ax_cross.set_yscale("log")
        ax_cross.set_ylim(850, 150)            # inverted: surface at the bottom
        ax_cross.yaxis.set_major_locator(FixedLocator([850, 700, 600, 500, 400, 300, 200]))
        ax_cross.yaxis.set_major_formatter(ScalarFormatter())
        ax_cross.yaxis.set_minor_formatter(NullFormatter())

        ax_cross.set_xlim(cross_lon_min, cross_lon_max)
        ax_cross.xaxis.set_major_locator(FixedLocator([-30, -20, -10, 0]))  # no edge ticks -> no overlap
        ax_cross.xaxis.set_major_formatter(FuncFormatter(lon_formatter))
        ax_cross.set_xlabel("")
        if left_labels:
            ax_cross.set_ylabel("Pressure [hPa]")
        else:
            ax_cross.tick_params(labelleft=False)
        ax_cross.set_title("42.5°N")
        return cf_pv

    except Exception as exc:  # noqa: BLE001
        ax_cross.text(0.5, 0.5, f"Error:\n{exc}", ha="center", va="center",
                      transform=ax_cross.transAxes, wrap=True)
        print(f"Error for {date_str}: {exc}")
        return None


# =========================================================================
# Build the 2 x N figure (maps on top, sections below)
# =========================================================================
n_cols = len(target_dates)
fig = plt.figure(figsize=(12, 5.4))
gs = fig.add_gridspec(
    2, n_cols + 1,
    width_ratios=[1.0] * n_cols + [0.04],
    height_ratios=[1.0, 1.05],
    left=0.06, right=0.94, bottom=0.08, top=0.93,
    hspace=0.32, wspace=0.12,
)

panel_labels = [f"({c})" for c in "abcdefghijklmnop"]
cf_map = cf_pv = None

for col, date_str in enumerate(target_dates):
    ax_map = fig.add_subplot(gs[0, col], projection=ccrs.PlateCarree())
    ax_cross = fig.add_subplot(gs[1, col])

    m = plot_map(ax_map, date_str, left_labels=(col == 0))
    c = plot_cross(ax_cross, date_str, left_labels=(col == 0))
    cf_map = m if m is not None else cf_map
    cf_pv = c if c is not None else cf_pv

    add_panel_label(ax_map, panel_labels[col])              # (a)-(d): maps
    add_panel_label(ax_cross, panel_labels[n_cols + col])   # (e)-(h): sections

# One colorbar per row, to the right
if cf_map is not None:
    cb1 = fig.colorbar(cf_map, cax=fig.add_subplot(gs[0, -1]))
    cb1.set_label("θ at 2 PVU [K]")
    cb1.set_ticks(theta_levels_fill)

if cf_pv is not None:
    cb2 = fig.colorbar(cf_pv, cax=fig.add_subplot(gs[1, -1]))
    cb2.set_label("PV [PVU]")
    cb2.set_ticks(pv_levels)

fig.savefig("pv_zonal_section_p1.png", dpi=300, bbox_inches="tight")
fig.savefig("pv_zonal_section_p1.pdf", bbox_inches="tight")
plt.show()
